## Import data

In [56]:
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

heart = pd.read_csv("heart.csv")
heart.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


## Encoding categorical variables

Encoding can be done before splitting the data, as it does not create data leakage. 

### Encoding gender

- Male = 0
- Female = 1

In [57]:
heart["Sex"] = heart["Sex"].map({"M":0, "F":1})
heart["Sex"].head()

0    0
1    1
2    0
3    1
4    0
Name: Sex, dtype: int64

### Encoding ChestPainType

- TA - 0
- ATA - 1
- ASY - 2
- NAP - 3


In [58]:
heart["ChestPainType"] = heart["ChestPainType"].map({"TA":0, "ATA":1, "ASY":2, "NAP":3})
heart["ChestPainType"].head()

0    1
1    3
2    1
3    2
4    3
Name: ChestPainType, dtype: int64

### Encoding RestingECG

- Normal - 0 
- ST - 1
- LVH - 2

In [59]:
heart["RestingECG"] = heart["RestingECG"].map({"Normal":0, "ST":1, "LVH":2})
heart["RestingECG"].head()

0    0
1    0
2    1
3    0
4    0
Name: RestingECG, dtype: int64

### Encoding ExerciseAngina

- N - 0
- Y - 1

In [60]:
heart["ExerciseAngina"] = heart["ExerciseAngina"].map({"N":0, "Y":1})
heart["ExerciseAngina"].head()

0    0
1    0
2    0
3    1
4    0
Name: ExerciseAngina, dtype: int64

### Encoding ST_Slope

- Up - 0  
- Flat - 1  
- Down - 2  

In [61]:
heart["ST_Slope"] = heart["ST_Slope"].map({"Up":0, "Flat":1, "Down":2})
heart["ST_Slope"].head()



0    0
1    1
2    0
3    1
4    0
Name: ST_Slope, dtype: int64

All categorical variables were encoded and turned into numbers. 

## Separating feature from the target

In [62]:
X = heart.drop("HeartDisease", axis = 1)
y = heart["HeartDisease"]

## Splitting into train set and test set 

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 0, test_size = 0.2, stratify = y)

X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, random_state = 0, test_size = 0.25, stratify = y_train) 

print("Training X:")
X_train.head()

Training X:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
359,53,0,3,105,0,0,0,115,0,0.0,1
438,63,0,3,130,0,0,1,111,1,0.0,1
843,58,1,0,150,283,1,2,162,0,1.0,0
683,44,1,3,118,242,0,0,149,0,0.3,1
903,56,0,1,130,221,0,2,163,0,0.0,0


In [64]:
print("Testing X:")
X_test.head()

Testing X:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
559,64,0,2,134,273,0,0,102,1,4.0,2
900,58,0,2,114,318,0,1,140,0,4.4,2
291,47,1,1,140,257,0,0,135,0,1.0,0
584,64,0,2,141,244,1,1,116,1,1.5,1
404,47,0,3,110,0,1,0,120,1,0.0,1


In [65]:
print("Validation X:")
X_valid.head()

Validation X:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
681,51,0,2,140,261,0,2,186,1,0.0,0
121,52,1,3,125,272,0,0,139,0,0.0,0
614,62,0,1,120,254,0,2,93,1,0.0,1
267,34,0,1,98,220,0,0,150,0,0.0,0
881,44,0,1,120,263,0,0,173,0,0.0,0


In [66]:
print("Training y:")
y_train.head()

Training y:


359    1
438    1
843    0
683    0
903    0
Name: HeartDisease, dtype: int64

In [67]:
print("Testing y:")
y_test.head()

Testing y:


559    1
900    1
291    0
584    1
404    1
Name: HeartDisease, dtype: int64

In [68]:
print("Validation y:")
y_valid.head()

Validation y:


681    0
121    0
614    1
267    0
881    0
Name: HeartDisease, dtype: int64

## Removing missing values

### Cholesterol

Some observations had cholesterol = 0, which is not realistic. So these values were treated as missing values.

As the total number of observations with missing cholesterol values is large, it was decided to replace the 0 values with the median instead of just removing it. The median was calculated only from non zero cholesterol values in the training set. And the same median was then used for both the training and testing sets.

The median was used, because it is less sensitive to outliers than the mean, so the data is less likely to be distorted.


In [69]:
print("Quantity of missing values in X_train before replacement: ", (X_train["Cholesterol"] == 0).sum())
print("Quantity of missing values in X_test before replacement: ", (X_test["Cholesterol"] == 0).sum())
print("Quantity of missing values in X_valid before replacement: ", (X_valid["Cholesterol"] == 0).sum())

median_cholesterol = X_train.loc[X_train["Cholesterol"] != 0, "Cholesterol"].median()

X_train["Cholesterol"] = X_train["Cholesterol"].replace(0, median_cholesterol)
X_test["Cholesterol"] = X_test["Cholesterol"].replace(0, median_cholesterol)
X_valid["Cholesterol"] = X_valid["Cholesterol"].replace(0, median_cholesterol)

Quantity of missing values in X_train before replacement:  99
Quantity of missing values in X_test before replacement:  35
Quantity of missing values in X_valid before replacement:  38


In [70]:
print("Quantity of missing values in X_train after replacement: ", (X_train["Cholesterol"] == 0).sum())
X_train["Cholesterol"].head()

Quantity of missing values in X_train after replacement:  0


359    236
438    236
843    283
683    242
903    221
Name: Cholesterol, dtype: int64

In [71]:
print("Quantity of missing values in X_test after replacement: ", (X_test["Cholesterol"] == 0).sum())
X_test["Cholesterol"].head()

Quantity of missing values in X_test after replacement:  0


559    273
900    318
291    257
584    244
404    236
Name: Cholesterol, dtype: int64

In [72]:
print("Quantity of missing values in X_valid after replacement: ", (X_valid["Cholesterol"] == 0).sum())
X_test["Cholesterol"].head()

Quantity of missing values in X_valid after replacement:  0


559    273
900    318
291    257
584    244
404    236
Name: Cholesterol, dtype: int64

### RestingBP

There was just one observation with a missing RestingBP value. It was removed from the dataset.

In [73]:
print("Quantity of missing values in X_train before deletion: ", (X_train["RestingBP"] == 0).sum())
print("Quantity of missing values in X_test before deletion: ", (X_test["RestingBP"] == 0).sum())
print("Quantity of missing values in X_valid before deletion: ", (X_valid["RestingBP"] == 0).sum())

index_missing = X_train[X_train["RestingBP"] == 0].index
X_train = X_train.drop(index = index_missing)
y_train = y_train.drop(index = index_missing)

print("")
print("Quantity of missing values in X_train after deletion: ", (X_train["RestingBP"] == 0).sum())

Quantity of missing values in X_train before deletion:  1
Quantity of missing values in X_test before deletion:  0
Quantity of missing values in X_valid before deletion:  0

Quantity of missing values in X_train after deletion:  0


## Scaling numerical variables

In [74]:
columns = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
scale = StandardScaler()
X_train[columns] = scale.fit_transform(X_train[columns])
X_test[columns] = scale.transform(X_test[columns])
X_valid[columns] = scale.transform(X_valid[columns])

In [75]:
X_train[columns].head()

,Age,RestingBP,Cholesterol,MaxHR,Oldpeak
359,-0.069963,-1.554713,-0.114532,-0.843757,-0.858810
438,0.985255,-0.173316,-0.114532,-0.999435,-0.858810
843,0.457646,0.931801,0.762405,0.985470,0.093881
683,-1.019660,-0.836387,-0.002583,0.479513,-0.573003
903,0.246602,-0.173316,-0.394406,1.024389,-0.858810


In [76]:
X_test[columns].head()

,Age,RestingBP,Cholesterol,MaxHR,Oldpeak
559,1.090776,0.047707,0.575823,-1.349713,2.951953
900,0.457646,-1.057410,1.415443,0.129236,3.333030
291,-0.703094,0.379242,0.277291,-0.065362,0.093881
584,1.090776,0.434498,0.034734,-0.804837,0.570226
404,-0.703094,-1.278433,-0.114532,-0.649158,-0.858810


In [84]:
X_valid[columns].head()

,Age,RestingBP,Cholesterol,MaxHR,Oldpeak
681,-0.281007,0.379242,0.351924,1.919543,-0.85881
121,-0.175485,-0.449595,0.557164,0.090316,-0.85881
614,0.879733,-0.725875,0.221316,-1.699990,-0.85881
267,-2.074878,-1.941504,-0.413064,0.518433,-0.85881
881,-1.019660,-0.725875,0.389240,1.413586,-0.85881


## Result 

In [85]:
X_test.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
559,1.090776,0,2,0.047707,0.575823,0,0,-1.349713,1,2.951953,2
900,0.457646,0,2,-1.057410,1.415443,0,1,0.129236,0,3.333030,2
291,-0.703094,1,1,0.379242,0.277291,0,0,-0.065362,0,0.093881,0
584,1.090776,0,2,0.434498,0.034734,1,1,-0.804837,1,0.570226,1
404,-0.703094,0,3,-1.278433,-0.114532,1,0,-0.649158,1,-0.858810,1


In [86]:
X_train.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
359,-0.069963,0,3,-1.554713,-0.114532,0,0,-0.843757,0,-0.858810,1
438,0.985255,0,3,-0.173316,-0.114532,0,1,-0.999435,1,-0.858810,1
843,0.457646,1,0,0.931801,0.762405,1,2,0.985470,0,0.093881,0
683,-1.019660,1,3,-0.836387,-0.002583,0,0,0.479513,0,-0.573003,1
903,0.246602,0,1,-0.173316,-0.394406,0,2,1.024389,0,-0.858810,0


In [92]:
X_valid.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
681,-0.281007,0,2,0.379242,0.351924,0,2,1.919543,1,-0.85881,0
121,-0.175485,1,3,-0.449595,0.557164,0,0,0.090316,0,-0.85881,0
614,0.879733,0,1,-0.725875,0.221316,0,2,-1.699990,1,-0.85881,1
267,-2.074878,0,1,-1.941504,-0.413064,0,0,0.518433,0,-0.85881,0
881,-1.019660,0,1,-0.725875,0.389240,0,0,1.413586,0,-0.85881,0


In [93]:
y_test.head()

559    1
900    1
291    0
584    1
404    1
Name: HeartDisease, dtype: int64

In [94]:
y_train.head()

359    1
438    1
843    0
683    0
903    0
Name: HeartDisease, dtype: int64

In [98]:
y_valid.head()

681    0
121    0
614    1
267    0
881    0
Name: HeartDisease, dtype: int64

## Validation

In [99]:
print("Training sets size: ") 
print("X: ", X_train.shape)
print("y: ", y_train.shape)

print("Testing sets size: ") 
print("X: ", X_test.shape)
print("y: ", y_test.shape)

print("Validation sets size: ") 
print("X: ", X_valid.shape)
print("y: ", y_valid.shape)
print("")

Training sets size: 
X:  (549, 11)
y:  (549,)
Testing sets size: 
X:  (184, 11)
y:  (184,)
Validation sets size: 
X:  (184, 11)
y:  (184,)



In [100]:
print("Missing values in training sets:")
print("X: ", X_train.isna().sum().sum())
print("y: ", y_train.isna().sum().sum())

print("Missing values in testing sets:")
print("X: ", X_test.isna().sum().sum())
print("y: ", y_test.isna().sum().sum())

print("Missing values in validation sets:")
print("X: ", X_valid.isna().sum().sum())
print("y: ", y_valid.isna().sum().sum())

Missing values in training sets:
X:  0
y:  0
Missing values in testing sets:
X:  0
y:  0
Missing values in validation sets:
X:  0
y:  0


## Saving data

In [101]:
X_train.to_csv("Preprocessed_Data/X_train.csv", index = False)
X_test.to_csv("Preprocessed_Data/X_test.csv", index = False)
X_valid.to_csv("Preprocessed_Data/X_validation.csv", index = False)

y_train.to_csv("Preprocessed_Data/y_train.csv", index = False)
y_test.to_csv("Preprocessed_Data/y_test.csv", index = False)
y_valid.to_csv("Preprocessed_Data/y_validation.csv", index = False)